In [1]:
# ============================================================
# CELL 1: Imports
# ============================================================
import os
import json
from pathlib import Path
from datetime import datetime

import numpy as np
import rasterio

os.environ['MPLCONFIGDIR'] = f"/tmp/matplotlib_{os.environ.get('USER', 'user')}"

print("✓ Libraries imported")

✓ Libraries imported


In [3]:
ALIGNED_DATA_DIR = "/p/scratch/training2600/TeamGudnason/data/aligned_data"
OUTPUT_DIR       = "/p/scratch/training2600/TeamGudnason/training_data/patches_224"

PATCH_SIZE    = 224
STRIDE        = 224     # no overlap
MAX_PATCHES   = 500     # 224x224 patches are big, 500 per tile is plenty

NORMALIZATION    = 'percentile'
PERCENTILE_LOW   = 2
PERCENTILE_HIGH  = 98

SKIP_EXISTING = False
TILE_SELECTION = 'all'
TILE_LIST = []

print(f"Patch size:  {PATCH_SIZE}x{PATCH_SIZE} ({PATCH_SIZE * 10}m x {PATCH_SIZE * 10}m)")
print(f"Output dir:  {OUTPUT_DIR}")
print(f"Max patches per tile: {MAX_PATCHES}")

Patch size:  224x224 (2240m x 2240m)
Output dir:  /p/scratch/training2600/TeamGudnason/training_data/patches_224
Max patches per tile: 500


In [4]:
def find_aligned_pairs(data_dir, selection='all', tile_list=None):
    data_path = Path(data_dir)
    if not data_path.exists():
        print(f"❌ Data directory not found: {data_dir}")
        return []
    s2_files = sorted(data_path.glob("*_stacked.tif"))
    pairs = []
    for s2_path in s2_files:
        tile_name = s2_path.stem.replace('_stacked', '')
        if selection == 'list' and tile_list:
            if tile_name not in tile_list:
                continue
        corine_path = data_path / f"corine_aligned_{tile_name}.tif"
        if corine_path.exists():
            pairs.append({'s2_path': s2_path, 'corine_path': corine_path, 'tile_name': tile_name})
        else:
            print(f"⚠ Missing CORINE for: {tile_name}")
    return pairs

aligned_pairs = find_aligned_pairs(ALIGNED_DATA_DIR, TILE_SELECTION, TILE_LIST)
print(f"\n📂 Found {len(aligned_pairs)} aligned tile pair(s):")
for i, pair in enumerate(aligned_pairs, 1):
    print(f"  {i}. {pair['tile_name'][:60]}")


📂 Found 4 aligned tile pair(s):
  1. S2A_MSIL2A_20180523T095031_N0500_R079_T33TYN_20230903T233859
  2. S2A_MSIL2A_20181020T095031_N0500_R079_T33TYN_20230728T230408
  3. S2B_MSIL2A_20180408T095029_N0500_R079_T33TYN_20230906T162829
  4. S2B_MSIL2A_20180816T095029_N0500_R079_T33TYN_20230812T013012


In [5]:
def normalize_percentile(data, low_pct=2, high_pct=98):
    data = data.astype(np.float32)
    low_val  = np.percentile(data, low_pct)
    high_val = np.percentile(data, high_pct)
    normalized = (data - low_val) / (high_val - low_val + 1e-6)
    return np.clip(normalized, 0, 1), low_val, high_val

def normalize_data(data, method='percentile', **kwargs):
    if method == 'percentile':
        low_pct  = kwargs.get('low_pct', 2)
        high_pct = kwargs.get('high_pct', 98)
        normalized, low_val, high_val = normalize_percentile(data, low_pct, high_pct)
        params = {'method': 'percentile', 'low_pct': low_pct, 'high_pct': high_pct,
                  'low_val': float(low_val), 'high_val': float(high_val)}
    else:
        normalized = data.astype(np.float32)
        params = {'method': 'none'}
    return normalized, params

print("✓ Normalization functions defined")

✓ Normalization functions defined


In [6]:
def extract_patches(s2_path, corine_path, patch_size=224, stride=224,
                    max_patches=500, normalization='percentile', norm_kwargs=None):
    if norm_kwargs is None:
        norm_kwargs = {}
    try:
        with rasterio.open(s2_path) as s2_src, rasterio.open(corine_path) as corine_src:
            n_bands = s2_src.count
            height  = s2_src.height
            width   = s2_src.width
            if stride is None:
                stride = patch_size
            print(f"    Image size: {width} x {height} pixels")
            print(f"    Bands: {n_bands}")
            print(f"    Patch size: {patch_size}, Stride: {stride}")
            s2_data     = s2_src.read()
            corine_data = corine_src.read(1)

        print(f"    Applying {normalization} normalization...")
        s2_normalized, norm_params = normalize_data(s2_data, normalization, **norm_kwargs)

        patches, labels = [], []

        for y in range(0, height - patch_size + 1, stride):
            for x in range(0, width - patch_size + 1, stride):
                patch = s2_normalized[:, y:y+patch_size, x:x+patch_size]
                patch = np.transpose(patch, (1, 2, 0))  # (H, W, C)

                center_y = y + patch_size // 2
                center_x = x + patch_size // 2
                label = corine_data[center_y, center_x]

                if label < 1 or label > 44:
                    continue

                original_patch = s2_data[:, y:y+patch_size, x:x+patch_size]
                if np.any(original_patch == 0):
                    continue

                patches.append(patch)
                labels.append(label)

                if len(patches) >= max_patches:
                    break
            if len(patches) >= max_patches:
                break
            if y % 2000 == 0 and y > 0:
                print(f"    Progress: {len(patches):,} patches...")

        if len(patches) == 0:
            return None, None, None

        patches = np.array(patches, dtype=np.float32)
        labels  = np.array(labels, dtype=np.uint8)

        unique_labels, counts = np.unique(labels, return_counts=True)
        metadata = {
            's2_file':          Path(s2_path).name,
            'corine_file':      Path(corine_path).name,
            'patch_size':       patch_size,
            'stride':           stride,
            'n_patches':        len(patches),
            'n_bands':          patches.shape[3],
            'n_classes':        len(unique_labels),
            'label_distribution': {int(k): int(v) for k, v in zip(unique_labels, counts)},
            'extraction_date':  datetime.now().isoformat(),
            'patch_shape':      list(patches.shape),
            'bands':            ['B02', 'B03', 'B04', 'B08'],
            'normalization':    norm_params
        }
        return patches, labels, metadata

    except Exception as e:
        print(f"  ❌ Patch extraction failed: {e}")
        import traceback; traceback.print_exc()
        return None, None, None

print("✓ extract_patches defined")

✓ extract_patches defined


In [7]:
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

norm_kwargs = {'low_pct': PERCENTILE_LOW, 'high_pct': PERCENTILE_HIGH}
all_results = []

print("=" * 60)
print("EXTRACTING 224x224 PATCHES FOR LAB 6")
print("=" * 60)

for i, pair in enumerate(aligned_pairs, 1):
    print(f"\n[{i}/{len(aligned_pairs)}] {pair['tile_name'][:50]}...")

    output_npz   = Path(OUTPUT_DIR) / f"patches_{pair['tile_name']}_data.npz"
    metadata_file = Path(OUTPUT_DIR) / f"patches_{pair['tile_name']}_metadata.json"

    if SKIP_EXISTING and output_npz.exists():
        print("  ⏭ Skipped")
        all_results.append({'status': 'skipped'})
        continue

    patches, labels, metadata = extract_patches(
        pair['s2_path'], pair['corine_path'],
        patch_size=PATCH_SIZE, stride=STRIDE,
        max_patches=MAX_PATCHES,
        normalization=NORMALIZATION,
        norm_kwargs=norm_kwargs
    )

    if patches is None:
        print("  ❌ No patches extracted")
        all_results.append({'status': 'failed'})
        continue

    np.savez_compressed(output_npz, patches=patches, labels=labels)
    with open(metadata_file, 'w') as f:
        json.dump(metadata, f, indent=2)

    print(f"  ✅ {len(patches):,} patches, {metadata['n_classes']} classes")
    all_results.append({'status': 'success', 'n_patches': len(patches)})

print("\n" + "=" * 60)
print(f"Done. Output: {OUTPUT_DIR}")
total = sum(r.get('n_patches', 0) for r in all_results)
print(f"Total patches: {total:,}")

EXTRACTING 224x224 PATCHES FOR LAB 6

[1/4] S2A_MSIL2A_20180523T095031_N0500_R079_T33TYN_20230...
    Image size: 10980 x 10980 pixels
    Bands: 4
    Patch size: 224, Stride: 224
    Applying percentile normalization...
  ✅ 500 patches, 18 classes

[2/4] S2A_MSIL2A_20181020T095031_N0500_R079_T33TYN_20230...
    Image size: 10980 x 10980 pixels
    Bands: 4
    Patch size: 224, Stride: 224
    Applying percentile normalization...
  ✅ 500 patches, 18 classes

[3/4] S2B_MSIL2A_20180408T095029_N0500_R079_T33TYN_20230...
    Image size: 10980 x 10980 pixels
    Bands: 4
    Patch size: 224, Stride: 224
    Applying percentile normalization...
  ✅ 500 patches, 18 classes

[4/4] S2B_MSIL2A_20180816T095029_N0500_R079_T33TYN_20230...
    Image size: 10980 x 10980 pixels
    Bands: 4
    Patch size: 224, Stride: 224
    Applying percentile normalization...
  ✅ 500 patches, 18 classes

Done. Output: /p/scratch/training2600/TeamGudnason/training_data/patches_224
Total patches: 2,000


In [8]:
import numpy as np
d = np.load('/p/scratch/training2600/TeamGudnason/training_data/patches_224/patches_S2A_MSIL2A_20180523T095031_N0500_R079_T33TYN_20230903T233859_data.npz')
print('Shape:', d['patches'].shape)
print('Labels shape:', d['labels'].shape)
print('Unique labels:', np.unique(d['labels']))

Shape: (500, 224, 224, 4)
Labels shape: (500,)
Unique labels: [ 2  3  7 11 12 15 16 18 20 21 23 24 25 26 29 35 40 41]


In [9]:
import sys
!{sys.executable} -m pip install terratorch

  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
  Installing build dependencies ... one
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 592.7/592.7 kB 11.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 859.3/859.3 kB 24.3 MB/s  0:00:00
Using cached tqdm-4.67.3-py3-none-any.whl (78 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 129.0 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 129.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 645.5/645.5 kB 13.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 118.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 94.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 112.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 530.7/530.7 MB 64.7 MB/s  0:00:05m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━